# Module 5: RAG Pipeline with Azure DocumentDB

**Time**: ~60 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string in Step 0, then run each cell in order. You will load RAG chunks, create vector and BM25 search indexes, retrieve context with vector and hybrid search, and assemble a grounded prompt for a chat model.

The final cell prints the prompt that your application would send to Azure OpenAI or Azure AI Foundry. The lab keeps the model call out of scope so the database retrieval mechanics are visible.


## Step 0: Connect to Azure DocumentDB

This cell installs `pymongo` if needed, asks for a connection string if no environment variable is set, and opens the `docdbworkshop.rag_chunks` collection.

In [ ]:
import importlib.util, subprocess, sys, os, getpass
if importlib.util.find_spec("pymongo") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymongo"])
from pymongo import MongoClient

connection_string = os.environ.get("DOCUMENTDB_CONNECTION_STRING") or getpass.getpass("Paste Azure DocumentDB connection string: ")
client = MongoClient(connection_string)
db = client["docdbworkshop"]
chunks = db["rag_chunks"]
print(db.command({"ping": 1}))

## Step 1: Load RAG chunks

Each chunk stores source metadata, text, tags, and an embedding in the same Azure DocumentDB document. Production apps would generate these embeddings during ingestion.

In [ ]:
chunks.drop()
rag_docs = [
    {"_id":"rag-001","sourceId":"search-module","title":"Vector search","chunk":"Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity.","url":"module-4-search","tags":["vector","search"],"embedding":[0.92,0.80,0.18]},
    {"_id":"rag-002","sourceId":"search-module","title":"Full-text search","chunk":"Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches.","url":"module-4-search","tags":["full-text","bm25"],"embedding":[0.20,0.12,0.94]},
    {"_id":"rag-003","sourceId":"search-module","title":"Hybrid search","chunk":"Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion.","url":"module-4-search","tags":["hybrid","rrf"],"embedding":[0.76,0.70,0.42]},
    {"_id":"rag-004","sourceId":"rag-module","title":"Grounded generation","chunk":"A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data.","url":"module-5-rag","tags":["rag","generation"],"embedding":[0.84,0.73,0.34]}
]
chunks.insert_many(rag_docs)
print("Loaded chunks:", chunks.count_documents({}))

## Step 2: Create retrieval indexes

The vector index supports semantic retrieval. The full-text search index supports keyword retrieval for exact terms such as `BM25`, `$search`, and `cosmosSearch`.

In [ ]:
db.command({
    "createIndexes": "rag_chunks",
    "indexes": [{
        "name": "idx_chunk_embedding_diskann",
        "key": {"embedding": "cosmosSearch"},
        "cosmosSearchOptions": {"kind": "vector-diskann", "dimensions": 3, "similarity": "COS", "maxDegree": 32, "lBuild": 64}
    }]
})
db.command({
    "createSearchIndexes": "rag_chunks",
    "indexes": [{
        "name": "idx_chunk_fts",
        "definition": {"mappings": {"dynamic": False, "fields": {"chunk": {"type": "string"}}}}
    }]
})

## Step 3: Retrieve context with vector search

A RAG app embeds the user question, then uses `cosmosSearch` to retrieve semantically similar chunks. This lab uses a sample vector so the notebook works without an external embedding service.

In [ ]:
question = "How does DocumentDB retrieve context for RAG?"
question_vector = [0.83, 0.74, 0.33]
vector_context = list(chunks.aggregate([
    {"$search": {"cosmosSearch": {"path": "embedding", "vector": question_vector, "k": 3}}},
    {"$project": {"_id": 1, "title": 1, "chunk": 1, "url": 1, "score": {"$meta": "searchScore"}}}
]))
vector_context

## Step 4: Retrieve context with hybrid search

Hybrid retrieval combines BM25 and vector search. This improves RAG recall when a question contains both natural language and exact database/operator terms.

In [ ]:
keyword_context = list(chunks.aggregate([
    {"$search": {"index": "idx_chunk_fts", "text": {"query": question, "path": "chunk"}}},
    {"$limit": 3},
    {"$project": {"_id": 1, "title": 1, "chunk": 1, "url": 1, "score": {"$meta": "searchScore"}}}
]))

def rrf(lists, k=60, top_n=3):
    docs = {}
    scores = {}
    for results in lists:
        for rank, doc in enumerate(results):
            doc_id = str(doc["_id"])
            docs[doc_id] = doc
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return [{**docs[doc_id], "rrfScore": score} for doc_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

hybrid_context = rrf([keyword_context, vector_context])
hybrid_context

## Step 5: Build the grounded prompt

The retrieved chunks are formatted into a context block. The system instruction tells the model to answer only from that context and to say when the context is insufficient.

In [ ]:
context_block = "

".join([
    f"[{i+1}] {doc['title']}
{doc['chunk']}
Source: {doc['url']}"
    for i, doc in enumerate(hybrid_context)
])

grounded_prompt = f"""You are a helpful assistant for an Azure DocumentDB workshop.
Answer the user's question using only the context below.
If the context does not contain the answer, say you do not know based on the provided context.

<context>
{context_block}
</context>

Question: {question}"""
print(grounded_prompt)